# RAG — Phase 2: Hybrid Retrieval, Re-ranking & Citation Enforcement

Builds on `phase-1.ipynb`. Per `RAG.md`, Phase-2 adds:

1. **Hybrid retrieval** — BM25 keyword search combined with the existing vector/semantic search
2. **Cross-encoder re-ranking** — re-scores the merged candidate set by evaluating query+chunk pairs together
3. **Citation enforcement** — decline to answer if retrieved chunks don't actually support a claim
4. **Versioned prompts** — stored in a config file, not hardcoded in a cell

This notebook does **not** re-ingest documents — it reconnects to the Chroma
collection Phase-1 already built (`chroma_db/`, in this same folder) and
reconstructs the chunk `Document` objects from it. If you haven't run
`phase-1.ipynb`'s ingestion yet, do that first.

## 1. Setup & reconnect to the Phase-1 vector store

Same environment setup as Phase-1 (Groq key, `USER_AGENT`), plus reopening the
existing persisted Chroma collection instead of rebuilding it from scratch.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
assert GROQ_API_KEY, "Set GROQ_API_KEY in a .env file in this project's root."
os.environ.setdefault("USER_AGENT", "rag-phase2-notebook/1.0")
print("Groq key loaded:", GROQ_API_KEY[:6] + "..." if GROQ_API_KEY else "MISSING")

Groq key loaded: gsk_0d...


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

PERSIST_DIR = "chroma_db"          # same store phase-1.ipynb built, same folder
COLLECTION_NAME = "rag_phase1"

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma(
    persist_directory=PERSIST_DIR,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
)

print(f"Reconnected to '{COLLECTION_NAME}' with {vectorstore._collection.count()} chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Reconnected to 'rag_phase1' with 15 chunks


## 2. Reconstruct chunk documents for BM25

BM25 is a keyword-matching algorithm — it needs the raw chunk text up front to
build its index, unlike the vector store, which only needs a query embedding
at search time. Chroma will hand back everything it has stored (`documents`
and `metadatas`), which is enough to rebuild the same `Document` objects
Phase-1 originally chunked, without re-running any loaders or the splitter.

In [3]:
from langchain_core.documents import Document

raw = vectorstore.get(include=["documents", "metadatas"])
chunks = [
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(raw["documents"], raw["metadatas"])
]

print(f"Reconstructed {len(chunks)} chunk documents from the vector store")
if chunks:
    print("Example source:", chunks[0].metadata.get("source"))

Reconstructed 15 chunk documents from the vector store
Example source: ..\data\RAG.md


## 3. Hybrid retrieval — BM25 + semantic search

Vector search finds chunks that are *semantically* similar, which can miss an
exact keyword or code identifier the person actually typed. BM25 is the
opposite: it's a pure keyword/term-frequency match, which can miss paraphrases
or synonyms. `EnsembleRetriever` runs both and merges their ranked results
(via reciprocal rank fusion), so a chunk that either method rates highly
surfaces near the top.

Weights (`0.4` BM25 / `0.6` vector) favor semantic search slightly — a
reasonable default; tune per-domain once you have eval data (Phase-3).

In [4]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

BM25_K = 5
VECTOR_K = 5

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = BM25_K

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": VECTOR_K})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6],
)

# quick manual check
test_query = "What does Phase-2 add to the RAG pipeline?"
hybrid_results = hybrid_retriever.invoke(test_query)
print(f"Hybrid retrieval returned {len(hybrid_results)} candidate chunks")
for i, d in enumerate(hybrid_results, 1):
    print(f"[{i}] {d.metadata.get('source')}: {d.page_content[:100]!r}")

C:\Users\smrut\AppData\Local\Temp\ipykernel_26236\116452065.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Hybrid retrieval returned 7 candidate chunks
[1] ..\data\RAG.md: 'RAG\n\nPhase-1 ingest documents;pdfs,mds,webpages->chunk them to pieces;500-800 tokens;100 tokens over'
[2] https://en.wikipedia.org/wiki/Retrieval-augmented_generation: 'LLMs with RAG are programmed to prioritize new information. This technique has been called "prompt s'
[3] https://en.wikipedia.org/wiki/Retrieval-augmented_generation: 'Retrieval-augmented generation - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain me'
[4] https://en.wikipedia.org/wiki/Retrieval-augmented_generation: 'Retro language model for RAG.  Each Retro block consists of Attention, Chunked Cross Attention, and '
[5] ..\data\resume.pdf: 'Smruti Ranjan Kodamasingh♂phone+91-9178763926\nSoftware engineer✉smrutiranjankodamasingh19@gmail.com\n'
[6] https://en.wikipedia.org/wiki/Retrieval-augmented_generation: 'Different data styles have patterns that correct chunking can take advantage of.\nHybrid se

## 4. Cross-encoder re-ranking

Hybrid retrieval casts a wider net (BM25_K + VECTOR_K candidates, with some
overlap). A cross-encoder re-scores each *(query, chunk)* pair jointly — more
accurate than either retriever alone, but too slow to run over the whole
vector store, which is why it only runs on this small candidate set. We keep
the top `RERANK_TOP_N` after scoring.

`cross-encoder/ms-marco-MiniLM-L-6-v2` is a small, well-tested SBERT
cross-encoder that runs locally on CPU — no extra API key, matching the
`sentence-transformers` embedding choice from Phase-1.

In [5]:
from sentence_transformers import CrossEncoder

RERANK_TOP_N = 5

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, docs: list[Document], top_n: int = RERANK_TOP_N) -> list[Document]:
    if not docs:
        return []
    pairs = [(query, d.page_content) for d in docs]
    scores = cross_encoder.predict(pairs)

    ranked = sorted(zip(docs, scores), key=lambda pair: pair[1], reverse=True)
    reranked_docs = []
    for doc, score in ranked[:top_n]:
        # keep the score visible for debugging / future citation-confidence use
        doc.metadata["rerank_score"] = float(score)
        reranked_docs.append(doc)
    return reranked_docs

reranked = rerank(test_query, hybrid_results)
print(f"Re-ranked down to top {len(reranked)}")
for i, d in enumerate(reranked, 1):
    print(f"[{i}] score={d.metadata['rerank_score']:.3f}  {d.metadata.get('source')}: {d.page_content[:80]!r}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\Users\smrut\miniconda3\envs\rag\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\smrut\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Re-ranked down to top 5
[1] score=4.643  ..\data\RAG.md: 'RAG\n\nPhase-1 ingest documents;pdfs,mds,webpages->chunk them to pieces;500-800 to'
[2] score=-3.417  https://en.wikipedia.org/wiki/Retrieval-augmented_generation: 'LLMs with RAG are programmed to prioritize new information. This technique has b'
[3] score=-6.287  https://en.wikipedia.org/wiki/Retrieval-augmented_generation: 'Type of information retrieval using LLMs\nRetrieval-augmented generation (RAG) is'
[4] score=-7.513  https://en.wikipedia.org/wiki/Retrieval-augmented_generation: 'Retrieval-augmented generation - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to co'
[5] score=-7.823  https://en.wikipedia.org/wiki/Retrieval-augmented_generation: 'Different data styles have patterns that correct chunking can take advantage of.'


## 5. Versioned prompts

Phase-1 hardcoded `PROMPT_TEMPLATE` as a Python string in a cell. Per
`RAG.md`, prompts move to a version-controlled config file instead, so a
prompt change is a diffable, reviewable edit — not buried in notebook output —
and you can roll back to an earlier version if a new one regresses answer
quality (something Phase-3's eval suite will be able to measure directly).

This writes `prompts.yaml` next to the notebook. `v1` is Phase-1's original
prompt, kept for comparison. `v2` adds the citation-enforcement contract used
below: a strict, machine-checkable rule for citing evidence and an explicit
"I cannot answer" signal the code can detect and act on, instead of just
hoping the model behaves.

In [6]:
import yaml

PROMPTS_PATH = "prompts.yaml"

prompts_config = {
    "active_version": "v2",
    "versions": {
        "v1": {
            "description": "Phase-1 baseline: asks for citations, no enforcement.",
            "template": (
                "You are a helpful assistant answering questions using only the context below.\n"
                "Cite the source number (e.g. [1]) after every claim you make. If the context does not "
                "contain enough information to answer, say so explicitly instead of guessing.\n\n"
                "Context:\n{context}\n\n"
                "Question: {question}\n\n"
                "Answer (with citations):"
            ),
        },
        "v2": {
            "description": "Phase-2: strict citation contract + explicit insufficient-context signal.",
            "template": (
                "You are a precise assistant that answers ONLY from the numbered context chunks below.\n"
                "Rules:\n"
                "1. Every factual sentence must end with a citation like [1] or [2] referencing the "
                "chunk number it came from.\n"
                "2. Never cite a chunk number that is not listed below.\n"
                "3. If the context does not contain enough information to answer the question, "
                "respond with exactly: INSUFFICIENT_CONTEXT\n"
                "   Do not guess, speculate, or use outside knowledge.\n\n"
                "Context:\n{context}\n\n"
                "Question: {question}\n\n"
                "Answer:"
            ),
        },
    },
}

# Only write the file if it doesn't already exist, so re-running this cell
# doesn't clobber manual edits you've made to prompts.yaml.
if not os.path.exists(PROMPTS_PATH):
    with open(PROMPTS_PATH, "w") as f:
        yaml.safe_dump(prompts_config, f, sort_keys=False)
    print(f"Wrote {PROMPTS_PATH}")
else:
    print(f"{PROMPTS_PATH} already exists — leaving it as-is")

with open(PROMPTS_PATH) as f:
    loaded_prompts = yaml.safe_load(f)

ACTIVE_VERSION = loaded_prompts["active_version"]
PROMPT_TEMPLATE = loaded_prompts["versions"][ACTIVE_VERSION]["template"]
print(f"Active prompt version: {ACTIVE_VERSION}")

Wrote prompts.yaml
Active prompt version: v2


## 6. Citation enforcement

Two checks run on every answer before it's returned to the caller:

1. **Insufficient-context signal.** If the model followed the v2 prompt's
   rule and returned `INSUFFICIENT_CONTEXT`, surface that as a clear decline
   rather than passing it through as if it were a real answer.
2. **Citation validity.** Scan the answer for `[n]` markers and confirm every
   `n` actually refers to one of the chunks that was in context. A citation
   pointing outside that range is a hallucinated source — Phase-1 had no way
   to catch this; Phase-2 does.

This is a structural check, not a semantic one — it confirms citations point
to *real* chunks, not that the cited chunk *actually supports* the sentence
it's attached to. True faithfulness scoring (does the claim follow from the
cited text) is what Phase-3's `ragas` evaluation is for.

In [7]:
import re

CITATION_PATTERN = re.compile(r"\[(\d+)\]")

def check_citations(answer: str, num_chunks: int) -> dict:
    cited_numbers = {int(n) for n in CITATION_PATTERN.findall(answer)}
    invalid = {n for n in cited_numbers if n < 1 or n > num_chunks}
    return {
        "cited_numbers": sorted(cited_numbers),
        "invalid_citations": sorted(invalid),
        "has_invalid_citations": bool(invalid),
    }

def enforce_citations(answer: str, num_chunks: int) -> dict:
    """Returns a dict describing whether the answer should be shown as-is,
    or replaced with a decline message, plus why."""
    if answer.strip() == "INSUFFICIENT_CONTEXT":
        return {
            "accepted": False,
            "reason": "model signaled insufficient context",
            "final_answer": "I don't have enough information in the retrieved documents to answer that.",
        }

    citation_check = check_citations(answer, num_chunks)
    if citation_check["has_invalid_citations"]:
        invalid = citation_check["invalid_citations"]
        return {
            "accepted": False,
            "reason": f"answer cited out-of-range chunk(s): {invalid}",
            "final_answer": "I couldn't verify the sources for this answer, so I'm declining rather than risk citing something unsupported.",
        }

    return {"accepted": True, "reason": None, "final_answer": answer}

## 7. Put it together — the Phase-2 query function

Hybrid retrieval → cross-encoder re-rank → versioned prompt → Groq generation
→ citation enforcement. This replaces Phase-1's `answer_query`.

In [8]:
from langchain_groq import ChatGroq

GROQ_MODEL = "openai/gpt-oss-120b"   # see phase-1.ipynb note on Groq's Llama-model access change

llm = ChatGroq(model=GROQ_MODEL, temperature=0, groq_api_key=GROQ_API_KEY)

def format_context(docs: list[Document]) -> str:
    lines = []
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source", "unknown")
        lines.append(f"[{i}] (source: {src})\n{d.page_content}")
    return "\n\n".join(lines)

def answer_query_v2(question: str, verbose: bool = False) -> dict:
    candidates = hybrid_retriever.invoke(question)
    top_docs = rerank(question, candidates, top_n=RERANK_TOP_N)

    if not top_docs:
        return {
            "answer": "No relevant documents found.",
            "sources": [],
            "accepted": False,
            "reason": "no chunks retrieved",
        }

    context = format_context(top_docs)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    raw_answer = llm.invoke(prompt).content

    enforcement = enforce_citations(raw_answer, num_chunks=len(top_docs))

    if verbose:
        print("--- raw model output ---")
        print(raw_answer)
        print("--- enforcement result ---")
        print(enforcement)

    return {
        "answer": enforcement["final_answer"],
        "sources": [d.metadata.get("source") for d in top_docs] if enforcement["accepted"] else [],
        "accepted": enforcement["accepted"],
        "reason": enforcement["reason"],
    }

## 8. Test scenarios

Same spirit as Phase-1's §9 — manual sanity checks, not the automated
faithfulness suite (that's still Phase-3). These specifically exercise what's
*new* in Phase-2: hybrid retrieval, re-ranking, and enforcement.

### 8.1 A real, answerable question

Confirms the full v2 pipeline (hybrid retrieval → rerank → v2 prompt →
enforcement) still produces a normal accepted answer with citations, end to
end.

In [9]:
result = answer_query_v2("What does Phase-2 add to the RAG pipeline?", verbose=True)
print("\nFINAL:")
print(result)

--- raw model output ---
Phase‑2 augments the pipeline with a hybrid retrieval step that combines traditional BM25 keyword search with vector‑based semantic search, adds a cross‑encoder re‑ranker (e.g., using cohere or SBERT) to re‑score the initially retrieved chunks as query‑chunk pairs, and introduces citation enforcement so the system will explicitly refuse to answer when the retrieved chunks do not substantiate the response; it also stores prompts in version‑controlled configuration files [1].
--- enforcement result ---
{'accepted': True, 'reason': None, 'final_answer': 'Phase‑2 augments the pipeline with a hybrid retrieval step that combines traditional BM25 keyword search with vector‑based semantic search, adds a cross‑encoder re‑ranker (e.g., using\u202fcohere or\u202fSBERT) to re‑score the initially retrieved chunks as query‑chunk pairs, and introduces citation enforcement so the system will explicitly refuse to answer when the retrieved chunks do not substantiate the response

### 8.2 Out-of-scope question — enforcement should decline

Same off-topic test as Phase-1 §9.3, but now the decline is *enforced* by
code (via the `INSUFFICIENT_CONTEXT` signal), not just hoped for by the
prompt wording alone.

In [10]:
result = answer_query_v2("What is the capital of Mongolia?", verbose=True)
print("\nFINAL:")
print(result)
assert result["accepted"] is False, "Expected enforcement to decline this answer"
print("\nEnforcement correctly declined an out-of-scope question.")

--- raw model output ---
INSUFFICIENT_CONTEXT
--- enforcement result ---
{'accepted': False, 'reason': 'model signaled insufficient context', 'final_answer': "I don't have enough information in the retrieved documents to answer that."}

FINAL:
{'answer': "I don't have enough information in the retrieved documents to answer that.", 'sources': [], 'accepted': False, 'reason': 'model signaled insufficient context'}

Enforcement correctly declined an out-of-scope question.


### 8.3 Simulated hallucinated citation — enforcement should catch it

Enforcement logic is tested directly here (bypassing the LLM) to confirm it
actually rejects an out-of-range citation, rather than relying on the model
happening to misbehave during a real run.

In [11]:
fake_answer_with_bad_citation = "RAG stands for Retrieval-Augmented Generation [1]. It was proposed for open-domain QA [7]."
result = enforce_citations(fake_answer_with_bad_citation, num_chunks=3)  # only 3 chunks existed, [7] is invalid
print(result)
assert result["accepted"] is False
print("\nEnforcement correctly rejected a hallucinated citation ([7] when only 3 chunks existed).")

{'accepted': False, 'reason': 'answer cited out-of-range chunk(s): [7]', 'final_answer': "I couldn't verify the sources for this answer, so I'm declining rather than risk citing something unsupported."}

Enforcement correctly rejected a hallucinated citation ([7] when only 3 chunks existed).


### 8.4 Hybrid retrieval vs. vector-only — does BM25 help on keyword-heavy queries?

Runs the same query through both retrievers side by side. BM25 tends to win
on queries with exact terms/identifiers (e.g. `"SBERT"`, `"ragas"`) that a
semantic embedding might blur into a more general match.

In [12]:
keyword_query = "ragas"   # an exact tool name from RAG.md's Phase-3 row

vector_only = vector_retriever.invoke(keyword_query)
hybrid = hybrid_retriever.invoke(keyword_query)

print("Vector-only top sources:", [d.metadata.get("source") for d in vector_only])
print("Hybrid top sources:     ", [d.metadata.get("source") for d in hybrid])

Vector-only top sources: ['https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', '..\\data\\RAG.md']
Hybrid top sources:      ['..\\data\\RAG.md', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation']
